Zadanie 9

Stwórz modele Movie i Actor z relacją Many-to-Many (aktorzy grają w wielu filmach, filmy mają wielu aktorów). Dodaj 5 filmów i 7 aktorów, przypisz aktorów do filmów.

Dataset: Wykorzystaj prawdziwe nazwy filmów i aktorów (np. "Inception" - "Leonardo DiCaprio")

Wymagania:
* Tabela asocjacyjna
* Dodanie danych z relacjami
* Zapytanie: "Jakie filmy ma aktor X?"
* Zapytanie: "Jacy aktorzy grają w filmie Y?"

In [1]:
from sqlalchemy import Column, Integer, String, Table, ForeignKey, create_engine
from sqlalchemy.orm import relationship, sessionmaker, declarative_base

import os
from dotenv import load_dotenv

load_dotenv()

Base = declarative_base()

movie_actor_association = Table(
    'movie_actor',
    Base.metadata,
    Column('movie_id', Integer, ForeignKey('movies.id'), primary_key=True),
    Column('actor_id', Integer, ForeignKey('actors.id'), primary_key=True)
)

class Movie(Base):
    __tablename__ = 'movies'

    id = Column(Integer, primary_key=True)
    title = Column(String(255))

    actors = relationship('Actor', secondary=movie_actor_association, back_populates='movies')

    def __repr__(self):
        return f"<Movie(title='{self.title}', actors={len(self.actors)})>"

class Actor(Base):
    __tablename__ = 'actors'

    id = Column(Integer, primary_key=True)
    name = Column(String(255))

    movies = relationship('Movie', secondary=movie_actor_association, back_populates='actors')

    def __repr__(self):
        return f"<Actor(name='{self.name}', movies={len(self.movies)})>"


db_url = os.getenv("MOVIES_DB_URL")

engine = create_engine(db_url)
Base.metadata.create_all(engine)

Session = sessionmaker(bind=engine, expire_on_commit=False)
session = Session()

actor1 = Actor(id=1, name="Tom Hanks")
actor2 = Actor(id=2, name="John Travolta")
actor3 = Actor(id=3, name="Bruce Willis")
actor4 = Actor(id=4, name="Samuel L. Jackson")
actor5 = Actor(id=5, name="Leonardo DiCaprio")
actor6 = Actor(id=6, name="Kurt Russell")
actor7 = Actor(id=7, name="Uma Thurman")

movies = [
    Movie(id=1, title="The Green Mile", actors=[actor1]),
    Movie(id=2, title="Catch Me If You Can", actors=[actor5, actor1]),
    Movie(id=3, title="Titanic", actors=[actor5]),
    Movie(id=4, title="Pulp Fiction", actors=[actor2, actor4, actor3, actor7]),
    Movie(id=5, title="The Hateful Eight", actors=[actor4, actor6])
]

session.add_all(movies)
session.commit()

def get_actors_by_movie(movie_title):
    movie = session.query(Movie).filter(Movie.title.ilike(movie_title.strip())).first()
    return [actor.name for actor in movie.actors] if movie else []

def get_movies_by_actor(actor_name):
    actor = session.query(Actor).filter(Actor.name.ilike(actor_name.strip())).first()
    return [movie.title for movie in actor.movies] if actor else []

print(f"Filmy, w których gra Tom Hanks: {get_movies_by_actor('Tom Hanks')}")
print(f"Aktorzy grający w filmie Pulp Fiction: {get_actors_by_movie('Pulp Fiction')}")
print(f"Filmy, w których gra Samuel L. Jackson: {get_movies_by_actor('Samuel L. Jackson')}")
print(f"Filmy, w których gra Leonardo DiCaprio: {get_movies_by_actor('Leonardo DiCaprio')}")

session.close()

Filmy, w których gra Tom Hanks: ['The Green Mile', 'Catch Me If You Can']
Aktorzy grający w filmie Pulp Fiction: ['John Travolta', 'Samuel L. Jackson', 'Bruce Willis', 'Uma Thurman']
Filmy, w których gra Samuel L. Jackson: ['Pulp Fiction', 'The Hateful Eight']
Filmy, w których gra Leonardo DiCaprio: ['Catch Me If You Can', 'Titanic']
